# Shapes from 60 labels: macro F1 on a CPU budget

An open-ended image problem with few labels, in the spirit of 2026 Round 2 Problem 4 (adapted: new classes, image size, label count and interface; lesson u19-l05).

Each image is a small RGB picture of one shape on a light background: **0** disk, **1** triangle, **2** rectangle, **3** cross. Position, size and rotation are random, triangles are not equilateral, rectangles can be long and thin, the shape and background colors are random, and every pixel has sensor noise. Only 60 of the 1,200 training images have labels. Predict the shape.

```python
def fit_predict(X, y, X_test):
    # Your code
    return labels
```

- `X`: `uint8` array of shape `(1200, 3, 28, 28)` (channels first, values 0 to 255).
- `y`: `int64` array of shape `(1200,)`. 60 entries are labels 0 to 3; the other 1,140 are `-1`, meaning hidden. The labeled rows are scattered, not the first 60.
- `X_test`: `uint8` array of shape `(m, 3, 28, 28)`.
- Train inside the function, then return a Python `list` of `m` Python `int`s (0 to 3), the predicted shapes in the order of `X_test`. Do not modify the inputs.

**Scoring.** [[macro F1]] over the four classes on a public test of 600 images, shown by Run, and a hidden test of 600 images from the same distribution. The grader passes both test sets in one call, shuffled together. You pass with macro F1 at least **0.90** on the hidden test (0.89 on the public test). Measured for reference: always predicting the most common label scores 0.10; a 1-nearest-neighbor classifier on raw pixels 0.32; a small CNN trained on the 60 labels with suitable preprocessing and augmentation about 0.93.

**Unlabeled images.** You may use them, by code only (the contest rule bans labeling by eye). They are not a free gain: giving every unlabeled image the guess of a weakly trained model and retraining on those guesses scored 0.63 here. Compare any semi-supervised step against the supervised model on held-out true labels before keeping it.

**Restrictions.** You may import `numpy`, `torch`, `math`, `sklearn` and `scipy`. No pretrained models and no `torchvision`. Seed your training so that it is reproducible.

**Constraints.** The whole call (training and prediction) must take at most 50 seconds on Colab's CPU. The grader runs PyTorch on one thread of a faster CPU and allows 25 seconds there.

**Data for your own experiments.** `make_data.py` in this folder writes `train.npz` (the same 1,200 images and labels the grader uses) and `val.npz` (300 labeled images from the same distribution).


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile submission.py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def fit_predict(X, y, X_test):
    # Your code
    return labels


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import submission as submission
importlib.reload(submission)
# e.g. submission.fit_predict(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9XA932zaS/yqo+64hE5mR7CSXKk1vndRtc00avzh9m3uyHkuRkMw1RXIJMrbq9Xe/3wwAEqRkN0n3LvvWkkBg/mFmMDMY9novTfamYq+ZfLtfjp/sjcTeWtYRhq63n9RpnUkaOz2PSqnEsirW4slYZNFCZmoq1lFcFeLHiShyEYmXJ7+JRZOsZE1rM6lUkVuA2fgxo0rzppYKo4eP8TNJl8s0brJ6g5FHGJB5XdH3vWVah2UlkzRmYOsiaTIZ5tGayVHNYp0qlQI+HkZZVlzKJEzXZVHVBHy2lzfrcsMsFFV8ziCimj/VRSajiheqOMWkOb4tojyXCa8kHgl1XUUpjVkYH1OLD09ytSyqtayUXR1mRVES6mWUKUmTpKrDOl3LoqlDejA5GNNwtNIEanCBkus0VE0pKwxobOZJnOdmc6o0VgGLOlxOjDRkluaroChlHso80QvbYUIbmp0g+lTRVDELLkqispbJ/sH44Mn+u4P9k0e0sIyq+gCP66qRNyMSfsZ7dL13DklhH4OSN+WM//ezzECuEvV5VIuyKhaZXItVFSU0qPdAeL+zqpjldvT+735wlp/lR2a6SJWIi7zGrj9s8rRWD7+jj+8fGqj4XabJ9w/1bBARiFe1SOQSGwP8kLBYNnlcQ1xKkG4kPBje17TV0YXEN3mWd/oitCaJ3zH0eyCOPxy9OXl9fCqyVNXEktRgCZgS6ry4zEm33zW58GhfQJX/TJwSuFpUDdDKj7La8CKw9mOUZk2FldAdJQX9FJdpfQ7jADwVraS4rNIaDAvoj3gh86lQ0UZcErmXkcLTIl+NRE5QRQIdFudpkmB6mpdNrUh6ehdyI9K4KDftD9p3kq/5yTYgADUvafRr8bKA4ZKZKhFVUigym4pklRMp95R4cyBOqmIkVCGyIo4yggiFUmLdQCgLSUKFiAqy2hSPs5TksAD/59jKZRTDUgLCdBCMaXMjsWpo/z2NWQFWJvfr80pGiVClxIZhVlU0q/NsA23JlgK0gIq4qEjQlSSh1yJdEoH4yQZu+DDGRUI5PTk+/iH88ejl+7fvxHPCTgyf5XEWATttg3d8FcuSNMWfnuUC/0o80rOgUJC4TDzoYjISa7Wyc4A3L0jIeWJG6F+3uR7N7YCEeeld2bVwZM4au0PkSZzRJfhPc1VHeSy9q5F+HryXuSoq31nOaGXdVLm4ChL46/jc84O4bPCXt9nz9WTJbIpXjO64qorKgaJZdmDlZRCpqKqiDeju+FDk7b1VUY+glDn9hX4+P9uDZsNXn+1ZyhRkXTdlJj3iHPP9gJcaWlimmGMnESx/JJZne9cE8AYbrjQuca3gd+RVKWN4J3HtzL8Buo6yOCvUXZSNRFUX2fOJ3H8yEpH5+tTSC8u6BMktsZGqN2B0mRUREUbjjLP/wGFmpfkjli7111vY0QyY6f4u3uwjZq9FgA1JFdxbWmM16MgyzxfwFKSF7rNL88xFT54URxZ5wpzWpPmyBW4UmfY7y4wQIQwjLvpjxEV/XMWTVQWJsSCIuHV0RR/RQnkrsS8ufd8n2GA0/UMKxARSjMn4dphKX07s6MTHKIN/EF4WVStyu5IUVlzjYxocrm783t7LfzZR9gla2TcqPZWIzhPW9JHPEgURUV1X5vnZHm9HB2SHujgK0s35P1cLVzWY/tDIAcRtGZMr1RYC7cp0SDGUgahl5jsYcMbXePRV1SONZmGotxmpCtlEeDvom7p1PxihsyPdki36aRSiMID9IOSILwx75CCq4Z0zWO/ZSfcs1D6locqjEid5/f/rmrMil5/plIHSauUVqSSf7Xs+jsSErbdPj0ex8YgDF9+lp89cjzoC6CVSls/fI9JzFMwQ+B7iG5K3DcDvHSE0FBBMfnbVEzy2xYtGYvHlco8+Se56irYLxvelYo9I7AxH9V2BQRRpJMpb+Lt8DZZ3fmbn8tZPLMxX3tu+ZRMHI+3toNm5u1cWDkNwgpciXDcwCkQ33jIfiftwp7CM+/cvLumbpUSHji9xAohlzpiXFKASD4hxEAWuYF5RjpCxWjVrBOU4QkSZRbEMbNTJpz7ZE/zirLOsyOd4NqIFhHGuZyI6o8Mj94YE6ce0JIUig2Hl01KJUEZWEc63P9LS02sYR0/B2Z+0ymWcyAYpDnQxkRTgp8uUIksKmTtevr5OxQMxuXkmLovqQmespLOCtg/xaOsvjZBBfCdhCuLT3FOSQkFQ9alSfuaKGN7VDbl3RNKcyeI8VIi9wQEJiGPde+1KBO86hyJUbwGiohQMCdJaEUsch7+JYhEDAS0BkzrqpmyFA3sKDpBd7Ov8kIY5BkdYnlIC1ZQwJ1FfprEkgJBQihO6WBoLBs1WoopViEaWiJfo+CYIWNHgWSDe04O0wjCYye4RCjoXNDKSsEx6alWPKZoEhADZ5TJkMLKy3uZPVCmpdy9GjFKbcMTw+1yYTRQPhZs2dCoZkipWZA3egat32ESg+e65gTRwRQvkNBfdUHQwEhcHwNb3j0zyaDDY4+TPRcF0kiAOWAwHzjCLARmbl9ARe4s43HjTZYjtqC4K1hsk/0k9DQ6WN4jyoWXymcm6WiVmZYLZMSDR5JTPXxvRmHWcQyKxs1u84x+jRGaJNHgJ1VpRAKuTzGuzTwyIrONOA63yFaxTJnbDugQHO5kUa0h7GUH9wnbiGZdXGhVFaRFyhaFX5fiJRmABoKwrHqTsNM7TLCGri6kygoQ5JVugmf99+vZX0Jqz5ag6AYVsq+WmPsfI/ith8YjvTIkjTNLqe/FdhwMP8RtW+S8eq7/fSvlhbE76j429qrN00Q7pD4wETZ1m3XDRfv2HKroiQRnV5+5yla7yqFumNmpQXrA/KpwLiyi+aEegxTkVC3StgWtr0I3W3WHgIq33ufgmJsGTYGJiOjWlOtGa6oUjdinILszhY51fGaWQ8uostziCZZpBo+1P72wPZBeVpAzQVFoQiQb3f8svcqrhqCKjikrBFQAgTEvateC+iRR/Pn53DMsxsghO8OmFIZXBwtAP4LhoOaK6MqpA2VkOqQQ0OYC2yqr2xjAGhA8ExgfAXbUvrgWJr3FW/zOaiuNH4wOuhL3++9H/nIYvjn799fgHUHB9theaOmYYEjNgzO6n/imvZGy+Ic7X3+JiXYLWs72bs/z127cn4a9vfzg+BTgPuhL8WMBW6cvfobhSfz1SmzxuH7xGDPkSMPSvU+n8+AFBZvfrJ5nT+VxUx1el75ZY3utazJ9VWcIImd7aIx1r1vA3OLZka7GcKlpAbjWCIps4jM9lfKE8VcW0xXXUP2/faXunaBiHFcmxuNDCWasVBPMMbpEiQF3jY7NR9b12dzgYMqVEOqio5sqq0jui9CE0m++KZutKIrBjKUFLkF+D0H4cerpBfn7FcSgV5eR2fDgzZE+hLVxFNtRPyU0qXm5S5CLXfuZaBvSZFzdT+o7ZSH5uDIGmLk5HAuQVwDHDTgbFcpLRbG4IXUTtOJ+TtT1yuBwaNoqBOeML2EOoa6VmnFQcISNsN01gKhSJVTpQ0IABQc/3KKOhoDtd0I0A2Lng2E1mASm2QcBVdUfm66heN5k7suTqCFw1+WYI/zLKLjzajMHR7YTpNF3rs04NhllFG8rSxIDJnW4fYKakgnjZi3gSBeVrKi3onwGwQZAUOXprRJ5wR5TMEfA1E6s3wp/uPhudvQiiJDE4/O3J3d448wJVZiltN+Gc7U/mOxZ2mxc0ZUJR9461bpKY3S3GH+H1htxgQ7FXLEdTeAf7nUHtkCWvMILE978gxaEEAW0gBUd0PQns0oDB0tuE90VLmc/PFfqRtZ0h7wOFYDrI0D4J6q+OR74LYJrcztXuSbswdofVECcbfhCVpcwTz1Z89KKzPe3ytHv/r768dmHhU/BFmr81p9hRszpSdAb5urYyWBAUZt6bqH6DkHFLp7UP+kziQJijkA5IChINqFvdv40CtJVoladRNgiaF/yjoPSUHazn4PH93d7lAZ0oEIwFdX1vJO5pINaiyADvAR/y3tU9KsXe7K4n7qKfShY7yFeW9g7Y1+0N4iLKFck2PjcnBIeC3QHB5WcZIYn9wJ/+M52NZpfRRu3rC1xkMogL0xYQKaWBVuTZZmRxInPpjhpPa4EuIplQioKOjcjkCqn5JYVNgmcGQeA7hxmCMpEUtCka8HnK+aXeBc85NL/hw9E5hTW55vCFGv5LeM5Z+o3oxYWOCgHDZ6sOKFDCyIfge8rfoTuA7P+VHc4Li6O9ku12GZT3mQ+1EYVFqePJfrWNpmsjG5jepzBMGSX04m/CwqaU1bEb4enIyWCYjeeucu9g/3NE0GG9i3l2blss8+iXMazDJFb2HqMYJjMmpa7kucw5iW1jR1717+XfcHZzR6a+rFKAyTYhR7IeQCDRDSmb6kf0b3O5z3QmUsVVylmFKKGqsO+VMDfRPCG119eUsMFcJc1TfFnfrzAtqKxi89YAeTO+12G98GQQhu0DpHwm2CTgiDSXfKgvCQ9AYKeWlH5KG/NR5teyYKLSSyqZUMGKvCzXJo3S4S+FY07U3huirWCFodl8ndaRb4RJNyZ8QyKd+xGK/m+uGesNzbfCXiMS99pcDJ52mOVSJhtVq4+zNkS0rNw682DupMRGUhSAPxftlEMbrkPvMU4FhyArokR5HhPxkNSGbIJLEXsED2cWVTI86+yQPhFEQ4w7wUqDq4nPxbZd6ISRhmZz+k2pnup+6sKMVlmTSZMm0sCvBbbAhT/roM2JnFtS0dbQKYIlE1QzTdWcTwv6xVYyd8NZm9LZeFZx3XELresluGrhsTipOUR5erp7ymstsSXPJYhOi+C0pvLJq7dWeLrIE+gPz/w6ffXT0et3b0YmSzdT4c5CLu7aW2DHl/V7nEiWB2NLSz875iaYrlgVUGdXJWF8ejM8UDoMsyyNsk4JRWXJfPX+1Zvjd+G746PXo5a6QZijSkna0y+FBTQacnsXV3aoOslXJg5HTp+ZVo2uMIdtclzVAF+z2EZnYDFCQu3Rn+FCWIyep2b0mBOGuVb8bZbYhmQVUA0o1MvI8/hfILhxvzZhyi69koRRRGMcc+3NNI/kgdmrsceti+KC7xTEtd2PaTCmCrB1XQbLi0jJtkKkayBdSezF6+PxeNIR8Fls3EaxuRhiMukmhjrECrNRJgi64zxqzXoL9C5BWb9CeKHPpFF8E+8hAX80Ho+n8y82ZG6L7NdxeMjt79Bz7F2xvdEELyP9yP/zze1klRRSB0262Q6nC4G4cfP2z2Rh9Xk2ebanm/7ICtsjo+0DbPk2fYR/bnsrx/hWt9mShmZmmXY/U9iyWZ6eQp7Btg7aIN64PTprKGjwkKgstWOnb+TbP0aVsjiCtJZr5XH3TN4/Fdir2nt/ukCJQIa39N2CF7wEXUJySEIYB8EzH8ek/VUDv6WVgqIV7iLILWsDh0vuOc0b6e5frA9Y7Q+nDGT2eDqH/+Z7YVBrqtSC3ePZngFtJzvY6ClVgynt2ZWmGH+rL31oaByMbz7nTmy78eELTp2/evKYy7ltr/xFLs1swqyTHFkqBd7bvRtbHrxbrGMPbePsrLMiX9l7maHPFl6a6y4zjuZHlEAso0rYm0HfdQIGOV9uDGraf5HhlmYKr3f0qnzWWfLvosXtM+D7iuQTz5AOVqvgBK+CFifebRe1I3G463gz4ezcJmRV69g++yhqF+hY9Tmljl4Xu94VlbJTWRRF5m0Rpp8RqLqnuQSu7oFrV9gE5dYjhdual8ImPNq7hSHlN3RRNrXXA5Tu6Jvd9rDo7nTNmw1T047HV1fpMtVHzuDlBuZhMnk0Fk3OI9RRso5W1HKmqD05ob5n+wJEIJwXFritHeyD67zIYyk87iun9wkocFJ01V5TlkrDZbPI0pixmVZvjrY1KhDaLJc6asEeIr2Dk2WvdrJ5T/Uqwfk8dblTgvRMd7t3jRZxhFRF95EsCqxieevWjJW9yhPgrbjU3drcD1NQ2nghwyRCsFFugsH1811N5v2WrtuuQUfcMjAatF+c5W0zPo5Pcwxq6bB7gpqc5a9fvXn1Hs8f43DY2U/wddveUXTXxy9PfnumW2MUuTXTZaNh9btAbMvCWf7+6N1Px+9HQn+GJ7+9eP3qJTCPg29hmOPg6bdEMtF68NRFzxsHPPpGqkyvoEtn+S+Y96hH5lgkqboYiQkUI42oQwfZE/mI2vw4FFAtujz97YSQ9Fik90VUtC7pbQ/Tdj/S1losayETqM6gmxuWoi68i5Qa3Ku8bXDPAfpU3Be/nejfm81IXF3RlU0ZrFdVmszGU4Qv+DOHpEDJA7D+WH91KWI+92NJ3gtaVFRJmkdUtaWLQJYIv99hvA55vnwVYIjeovEeB5DoBDvqi7v+fa1fFIDjTdKGWpLYfnTfLsONr0Yi3gyAV9T0BV2DR63wf3ylVuXntmuG6xzu/DFtw33iv0z9O2nRDXcaTEOX7RDcPojw9fK4ULB7H+i9zYYebMwDBR/VHgwfsXB/sNJM2F5pQLYhP20necKxc+yuAbAR9++DiweAzl++A4v8zZZ3nbUTZy00T289lbC926RCx5JzLunyOOkLyMYHvdNFn1U4i+hdFvyZjecgpgWAYFl8JybBwVToIxtxRYr0UfXP3i+ipoQOlKwDncgACG6z6oRLA07TFIfr5dXM4wZBX/yHOJxD7BgBqVjlfaRfG/61Lzx8G87c2JlNt467G7tWskN/3t8kLyGxfI/d88U39GvS+3Vgfu3YsgN3ywyrPfEEjx+Ti/r2scMlFSiirYmHPPGpPyDNtPg3PqlOxBSZoY88tNh9UXC5k5jJU8JxOMTRR3K5jaTS9yL9wd68xs7rFUjXVJ3kN1hOR/BTsH368PnSyIuuUvXcm7DWtLZME099+C0oIfkqbNqMXNK8c6LmuJSU1vX6zKyS3t5mxn7VeBnENxKRtyIV/gXnoHn8QcOQ67LeeITikKk+9bmhuMGqp27PbKdVuT8wfcfZk1JqZ+9owWro7lhXJgEHm7e4uixdndeCSuMrDlghqKyonGbEbaDjFuYOR0Fbt6TXRxYrP1DN2iN3MA6eTHfgvpCy7Py8zSUH/Rqfhj9d07zFajYdcaFX/52T2U5AzJoc7nLH0/UQBP19wAhzYIsyjwOC8RNztOCxeYvDBiTUug43h3yhA/UBu6N3Pc7S0iOYcNBsuCP69AOdHLQvIw0UwSj7h5HYODMw4ckjp3fKBp0hN+cYXaE1QN2q9ORgTDjHB098e4/Jgehtev2fPmUrbcc5LfdnT5BUMLvU+LxoajeSrqQB2RnDpvcuwWamn5NE9idbDGp+7I2Gfg/WQ6RSWo6Wk2FLUNzZyC+9tyNKc6ZA7bySvGnMHmWjv7ottrfO/Go4M+/N/OoOmJO2bYAOMBDzUNhvDwgj/uRcE3IH9EUQ9Lm/9d1LWeTYANrmSeHLo5c/Uz/j9Y3Telc1uizRvSWF3KyizIxEpddst6Tp8ZmdOu9pUF+7zKMIzyJXvZ6Qdh1O7PMFni+2n9uA7ENtrAKZE55SFOnNCOiHhU0TEV1yzW+3dh4e9tUzk7n3oS1L6kSJX1zk+35kVSFSmNCMWy70QzV4ONl5t/HnlSndNwjhB06W6H0wJjBqjQFM1jPmbm5Ghr3dd7e4O9WY4euqvQmfXDfpv77nprim+rHzJlJXbGZT+IV59yLbEhkBMitXcDuFbD79Qdf+rmztrkZ2l1iueXE/+8TpZ09r/VozFbvUORlB274+7rWve9eM3A7bxNLvv7rJksBO+5RKUx/qFhn2tQcjNZrbye1Z93LPiW4Xt72s1F7eR0UqzZjgX4x634HMzr6x/pj847VZd+OWG/pYuIBDhH5klkAGVynGJOiPOLF/YUAfSXAEXrck08vwjMa+Mt7xk9K7hmO6fzlsERG1buzTcpMQ5ufOiWZnG/vAIu4qoOGhj6LokqdO4Vrm5I5GeukTqnyR+yH4fWe67efaI3RQh3CdaBkZ6AsLld4XtZ7Wnk10cdOeWjTbXhjHa837HxKZvuchIPylZT1tXzOliClJAvj6eD0SFgICnF7VDNtvSklcy+3+0xzXy8k0eLS8QSYWx00VIUO+9kp+hWwTmajYNzPuMBpnywgdNZ3ny0b/JxWiukqvhFdRVaXmrhBEiM06V8IAk8kzU+7oih1OqYMLHZBpH0W8drURYkQ+1KvGsM7fzTQprq5vXfeW9l8R5S3WEci/Y4sXNHu4OU5xbwedfctzeWUm717cMcl1RnqXurb8ash7Nzf/CyxDECg=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
